# Datakontroll — siste artikler

Henter artikler fra SQLite og leser tilhørende tekst fra Obsidian-vaulten.
Juster parameterne i neste celle.

In [9]:
# ── Parametere ────────────────────────────────────────────────
ANTALL      = 3       # Antall siste artikler å vise
SØKETEKST   = ""       # Filtrer på tittel/URL (tom = ingen filter)
ELEMENT_ID  = None     # Hent én spesifikk artikkel (None = bruk ANTALL)

In [10]:
import sqlite3
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown

# Last .env fra repo-rot
repo_rot = Path("__file__").resolve().parent.parent
load_dotenv(repo_rot / ".env")

DB_STI    = repo_rot / os.getenv("DATABASE_STI", "data/monitor.db")
VAULT_ROT = Path(os.getenv("VAULT_ROT", ""))

print(f"Database : {DB_STI}  (finnes: {DB_STI.exists()})")
print(f"Vault    : {VAULT_ROT}  (finnes: {VAULT_ROT.exists()})")

Database : C:\Users\Audun\2026\claude\personalisert_monitor\data\monitor.db  (finnes: True)
Vault    : C:\Users\Audun\2026\claude\OBSIDIAN\monitor-evals  (finnes: True)


In [11]:
conn = sqlite3.connect(DB_STI)
conn.row_factory = sqlite3.Row

if ELEMENT_ID is not None:
    sql = "SELECT * FROM elementer WHERE id = ?"
    params = (ELEMENT_ID,)
elif SØKETEKST:
    sql = """
        SELECT * FROM elementer
        WHERE tittel LIKE ? OR url LIKE ?
        ORDER BY id DESC LIMIT ?
    """
    mønster = f"%{SØKETEKST}%"
    params = (mønster, mønster, ANTALL)
else:
    sql = "SELECT * FROM elementer ORDER BY id DESC LIMIT ?"
    params = (ANTALL,)

artikler = conn.execute(sql, params).fetchall()
print(f"Fant {len(artikler)} artikkel(er)")

Fant 3 artikkel(er)


In [12]:
for art in artikler:
    # ── Metadata ──────────────────────────────────────────────
    meta = dict(art)

    # Sammendrag (kan være tomt)
    sam = conn.execute(
        "SELECT tekst, prompt_versjon, opprettet FROM sammendrag WHERE element_id = ? ORDER BY id DESC LIMIT 1",
        (meta["id"],)
    ).fetchone()

    # Artikkeltekst fra vault
    tekst = "*(vault_sti er None)*"
    if meta["vault_sti"]:
        vault_fil = VAULT_ROT / meta["vault_sti"]
        if vault_fil.exists():
            tekst = vault_fil.read_text(encoding="utf-8")
        else:
            tekst = f"*(fil ikke funnet: {vault_fil})*"

    # ── Visning ───────────────────────────────────────────────
    display(Markdown(f"---\n## [{meta['tittel']}]({meta['url']})"))
    display(Markdown(
        f"| Felt | Verdi |\n|------|-------|\n"
        + "\n".join(
            f"| `{k}` | {v} |"
            for k, v in meta.items()
        )
    ))

    if sam:
        display(Markdown(f"**Sammendrag** *(prompt: {sam['prompt_versjon']}, {sam['opprettet']})*\n\n{sam['tekst']}"))
    else:
        display(Markdown("*Ingen sammendrag ennå*"))

    display(Markdown("### Artikkeltekst"))
    display(Markdown(tekst))

---
## [Using LLM-as-a-Judge For Evaluation: A Complete Guide –](https://hamel.dev/blog/posts/llm-judge/)

| Felt | Verdi |
|------|-------|
| `id` | 3 |
| `kilde_id` | 6 |
| `guid` | 0792a2d3-10fa-4927-9a65-c210f94c2c0a |
| `url` | https://hamel.dev/blog/posts/llm-judge/ |
| `tittel` | Using LLM-as-a-Judge For Evaluation: A Complete Guide – |
| `publisert` | None |
| `hentet` | 2026-04-22T22:40:10.375806+00:00 |
| `vault_sti` | artikler\0792a2d3-using-llm-as-a-judge-for-evaluation-a-complete-guide.md |
| `dead_letter` | 0 |

*Ingen sammendrag ennå*

### Artikkeltekst

---
element_id: 0792a2d3-10fa-4927-9a65-c210f94c2c0a
url: https://hamel.dev/blog/posts/llm-judge/
kildetype: manuell
---

# Using LLM-as-a-Judge For Evaluation: A Complete Guide –

Earlier this year, I wrote [Your AI product needs evals](https://hamel.dev/blog/posts/evals/). Many of you asked, “How do I get started with LLM-as-a-judge?” This guide shares what I’ve learned after helping over [30 companies](https://parlance-labs.com/) set up their evaluation systems.

## The Problem: AI Teams Are Drowning in Data

Ever spend weeks building an AI system, only to realize you have no idea if it’s actually working? You’re not alone. I’ve noticed teams repeat the same mistakes when using LLMs to evaluate AI outputs:

1. **Too Many Metrics**: Creating numerous measurements that become unmanageable.
2. **Arbitrary Scoring Systems**: Using uncalibrated scales (like 1-5) across multiple dimensions, where the difference between scores is unclear and subjective. What makes something a 3 versus a 4? Nobody knows, and different evaluators often interpret these scales differently.
3. **Ignoring Domain Experts**: Not involving the people who understand the subject matter deeply.
4. **Unvalidated Metrics**: Using measurements that don’t truly reflect what matters to the users or the business.

The result? Teams end up buried under mountains of metrics or data they don’t trust and can’t use. Progress grinds to a halt. Everyone gets frustrated.

For example, it’s not uncommon for me to see dashboards that look like this:

![](../ressurser/bilder/651b658f.png)

An illustrative example of a bad eval dashboard

Tracking a bunch of scores on a 1-5 scale is often a sign of a bad eval process (I’ll discuss why later). In this post, I’ll show you how to avoid these pitfalls. The solution is to use a technique that I call **“Critique Shadowing”**. Here’s how to do it, step by step.

## Step 1: Find The Principal Domain Expert

In most organizations there is usually one (maybe two) key individuals whose judgment is crucial for the success of your AI product. These are the people with deep domain expertise or represent your target users. Identifying and involving this **Principal Domain Expert** early in the process is critical.

**Why is finding the right domain expert so important?**

- **They Set the Standard**: This person not only defines what is acceptable technically, but also helps you understand if you’re building something users actually want.
- **Capture Unspoken Expectations**: By involving them, you uncover their preferences and expectations, which they might not be able to fully articulate upfront. Through the evaluation process, you help them clarify what a “passable” AI interaction looks like.
- **Consistency in Judgment**: People in your organization may have different opinions about the AI’s performance. Focusing on the principal expert ensures that evaluations are consistent and aligned with the most critical standards.
- **Sense of Ownership**: Involving the expert gives them a stake in the AI’s development. They feel invested because they’ve had a hand in shaping it. In the end, they are more likely to approve of the AI.

**Examples of Principal Domain Experts:**

- A **psychologist** for a mental health AI assistant.
- A **lawyer** for an AI that analyzes legal documents.
- A **customer service director** for a support chatbot.
- A **lead teacher or curriculum developer** for an educational AI tool.

NoteExceptions

In a smaller company, this might be the CEO or founder. If you are an independent developer, you should be the domain expert (but be honest with yourself about your expertise).

If you must rely on leadership, you should regularly validate their assumptions against real user feedback.

Many developers attempt to act as the domain expert themselves, or find a convenient proxy (ex: their superior). This is a recipe for disaster. People will have varying opinions about what is acceptable, and you can’t make everyone happy. What’s important is that your principal domain expert is satisfied.

**Remember:** This doesn’t have to take a lot of the domain expert’s time. Later in this post, I’ll discuss how you can make the process efficient. Their involvement is absolutely critical to the AI’s success.

## Step 2: Create a Dataset

With your principal domain expert on board, the next step is to build a dataset that captures problems that your AI will encounter. It’s important that the dataset is diverse and represents the types of interactions that your AI will have in production.

### Why a Diverse Dataset Matters

- **Comprehensive Testing**: Ensures your AI is evaluated across a wide range of situations.
- **Realistic Interactions**: Reflects actual user behavior for more relevant evaluations.
- **Identifies Weaknesses**: Helps uncover areas where the AI may struggle or produce errors.

### Dimensions for Structuring Your Dataset

You want to define dimensions that make sense for your use case. For example, here are ones that I often use for B2C applications:

1. **Features**: Specific functionalities of your AI product.
2. **Scenarios**: Situations or problems the AI may encounter and needs to handle.
3. **Personas**: Representative user profiles with distinct characteristics and needs.

### Examples of Features, Scenarios, and Personas

#### Features

| **Email Summarization** | Condensing lengthy emails into key points. |
| --- | --- |
| **Meeting Scheduler** | Automating the scheduling of meetings across time zones. |
| **Order Tracking** | Providing shipment status and delivery updates. |
| **Contact Search** | Finding and retrieving contact information from a database. |
| **Language Translation** | Translating text between languages. |
| **Content Recommendation** | Suggesting articles or products based on user interests. |

#### Scenarios

Scenarios are situations the AI needs to handle, (not based on the outcome of the AI’s response).

| **Multiple Matches Found** | User’s request yields multiple results that need narrowing down. For example: User asks “Where’s my order?” but has three active orders (#123, #124, #125). AI must help identify which specific order they’re asking about. |
| --- | --- |
| **No Matches Found** | User’s request yields no results, requiring alternatives or corrections. For example: User searches for order #ABC-123 which doesn’t exist. AI should explain valid order formats and suggest checking their confirmation email. |
| **Ambiguous Request** | User input lacks necessary specificity. For example: User says “I need to change my delivery” without specifying which order or what aspect of delivery (date, address, etc.) they want to change. |
| **Invalid Data Provided** | User provides incorrect data type or format. For example: User tries to track a return using a regular order number instead of a return authorization (RMA) number. |
| **System Errors** | Technical issues prevent normal operation. For example: While looking up an order, the inventory database is temporarily unavailable. AI needs to explain the situation and provide alternatives. |
| **Incomplete Information** | User omits required details. For example: User wants to initiate a return but hasn’t provided the order number or reason. AI needs to collect this information step by step. |
| **Unsupported Feature** | User requests functionality that doesn’t exist. For example: User asks to change payment method after order has shipped. AI must explain why this isn’t possible and suggest alternatives. |

#### Personas

| **New User** | Unfamiliar with the system; requires guidance. |
| --- | --- |
| **Expert User** | Experienced; expects efficiency and advanced features. |
| **Non-Native Speaker** | May have language barriers; uses non-standard expressions. |
| **Busy Professional** | Values quick, concise responses; often multitasking. |
| **Technophobe** | Uncomfortable with technology; needs simple instructions. |
| **Elderly User** | May not be tech-savvy; requires patience and clear guidance. |

### Generating Data

To build your dataset, you can:

- **Use Existing Data**: Sample real user interactions or behaviors from your AI system.
- **Generate Synthetic Data**: Use LLMs to create realistic user inputs covering various features, scenarios, and personas.

Often, you’ll do a combination of both to ensure comprehensive coverage. Synthetic data is not as good as real data, but it’s a good starting point. Also, we are only using LLMs to generate the user inputs, not the LLM responses or internal system behavior.

Regardless of whether you use existing data or synthetic data, you want good coverage across the dimensions you’ve defined.

**Incorporating System Information**

When making test data, use your APIs and databases where appropriate. This will create realistic data and trigger the right scenarios. Sometimes you’ll need to write simple programs to get this information. That’s what the “Assumptions” column is referring to in the examples below.

### Example LLM Prompts for Generating User Inputs

Here are some example prompts that illustrate how to use an LLM to generate synthetic **user inputs** for different combinations of features, scenarios, and personas:

| 1 | **Order Tracking** | Invalid Data Provided | Frustrated Customer | “Generate a user input from someone who is clearly irritated and impatient, using short, terse language to demand information about their order status for order number **#1234567890**. Include hints of previous negative experiences.” | Order number **#1234567890** does **not** exist in the system. |
| --- | --- | --- | --- | --- | --- |
| 2 | **Contact Search** | Multiple Matches Found | New User | “Create a user input from someone who seems unfamiliar with the system, using hesitant language and asking for help to find contact information for a person named ‘Alex’. The user should appear unsure about what information is needed.” | Multiple contacts named ‘Alex’ exist in the system. |
| 3 | **Meeting Scheduler** | Ambiguous Request | Busy Professional | “Simulate a user input from someone who is clearly in a hurry, using abbreviated language and minimal details to request scheduling a meeting. The message should feel rushed and lack specific information.” | N/A |
| 4 | **Content Recommendation** | No Matches Found | Expert User | “Produce a user input from someone who demonstrates in-depth knowledge of their industry, using specific terminology to request articles on sustainable supply chain management. Use the information in this article involving sustainable supply chain management to formulate a plausible query: {{article}}” | No articles on ‘Emerging trends in sustainable supply chain management’ exist in the system. |

### Generating Synthetic Data

When generating synthetic data, you only need to create the user inputs. You then feed these inputs into your AI system to generate the AI’s responses. It’s important that you log everything so you can evaluate your AI. To recap, here’s the process:

1. **Generate User Inputs**: Use the LLM prompts to create realistic user inputs.
2. **Feed Inputs into Your AI System**: Input the user interactions into your AI as it currently exists.
3. **Capture AI Responses**: Record the AI’s responses to form complete interactions.
4. **Organize the Interactions**: Create a table to store the user inputs, AI responses, and relevant metadata.

#### How much data should you generate?

There is no right answer here. At a minimum, you want to generate enough data so that you have examples for each combination of dimensions (in this toy example: features, scenarios, and personas). However, you also want to keep generating more data until you feel like you have stopped seeing new failure modes. The amount of data I generate varies significantly depending on the use case.

#### Does synthetic data actually work?

You might be skeptical of using synthetic data. After all, it’s not real data, so how can it be a good proxy? In my experience, it works surprisingly well. Some of my favorite AI products, like [Hex](https://hex.tech/) use synthetic data to power their evals:

> “LLMs are surprisingly good at generating excellent - and diverse - examples of user prompts. This can be relevant for powering application features, and sneakily, for building Evals. If this sounds a bit like the Large Language Snake is eating its tail, I was just as surprised as you! All I can say is: it works, ship it.” *[Bryan Bischof](https://www.linkedin.com/in/bryan-bischof/), Head of AI Engineering at Hex*

## Step 3: Direct The Domain Expert to Make Pass/Fail Judgments with Critiques

The domain expert’s job is to focus on one thing: **“Did the AI achieve the desired outcome?”** No complex scoring scales or multiple metrics. Just a clear **pass or fail** decision. In addition to the pass/fail decision, the domain expert should write a critique that explains their reasoning.

### Why are simple pass/fail metrics important?

- **Clarity and Focus**: A binary decision forces everyone to consider what truly matters. It simplifies the evaluation to a single, crucial question.
- **Actionable Insights**: Pass/fail judgments are easy to interpret and act upon. They help you quickly identify whether the AI meets the user’s needs.
- **Forces Articulation of Expectations**: When domain experts must decide if an interaction passes or fails, they are compelled to articulate their expectations clearly. This process uncovers nuances and unspoken assumptions about how the AI should behave.
- **Efficient Use of Resources**: Keeps the evaluation process manageable, especially when starting out. You avoid getting bogged down in detailed metrics that might not be meaningful yet.

### The Role of Critiques

Alongside a binary pass/fail judgment, it’s important to write a detailed critique of the LLM-generated output. These critiques:

- **Captures Nuances**: The critique allows you to note if something was mostly correct but had areas for improvement.
- **Guide Improvement**: Detailed feedback provides specific insights into how the AI can be enhanced.
- **Balance Simplicity with Depth**: While the pass/fail offers a clear verdict, the critique offers the depth needed to understand the reasoning behind the judgment.

**Why Write Critiques?:**

In practice, domain experts may not have fully internalized all the judgment criteria. By forcing them to make a pass/fail decision and explain their reasoning, they clarify their expectations and provide valuable guidance for refining the AI.

*“But my problem is complex!”* Trust me—starting simple forces you to focus on what truly matters. You can introduce more complexity later if needed.

### Examples of Good Critiques

To illustrate how simple pass/fail judgments combined with detailed critiques work in practice, here’s a table showcasing examples of user interactions with an AI assistant. The table includes both **passes** and **fails**, with critiques explaining why the AI received that judgment. In cases where the AI passed despite critical issues, the critique highlights these aspects and justifies why it still passed overall. For failed interactions, the critique explains the critical elements that led to the failure.

| **User:** “I need to cancel my flight booking for next week.”   **AI:** “Your flight booking for next week has been canceled. A confirmation email has been sent to you.” | **Pass** | The AI successfully executed the user’s request by canceling the flight and sending a confirmation, achieving the primary goal. **However**, it should have asked for confirmation before proceeding to prevent accidental cancellations. Despite this critical oversight, the AI fulfilled the user’s request, so it **passes**, but adding a confirmation step would improve user safety. |
| --- | --- | --- |
| **User:** “What’s the capital of France?”   **AI:** “The capital of France is Paris.” | **Pass** | The AI correctly answered the user’s question, providing accurate information promptly. **Although** the response is sufficient, the AI could enhance the interaction by offering additional interesting facts or asking if the user has more questions. Despite this opportunity for added engagement, the AI met the user’s primary need, so it **passes**. |
| **User:** “I can’t log into my account; it says my password is incorrect.”   **AI:** “Please reset your password by clicking ‘Forgot Password’.” | **Fail** | The AI provided a generic solution without acknowledging the user’s frustration or checking for other issues like account lockout. It failed to offer empathetic support or additional assistance. Due to the lack of personalized help and poor user experience, it **fails**. |
| **User:** “Book a table for two at an Italian restaurant tonight at 7 PM.”   **AI:** “Table for two at Luigi’s Italian Restaurant booked for tonight at 7 PM. Confirmation number is 12345.” | **Pass** | The AI efficiently completed the booking as requested, providing all necessary details and a confirmation number. **However**, it could have asked if the user had seating preferences or dietary restrictions. The AI accomplished the primary task successfully, so it **passes**, with minor opportunities for enhanced service. |
| **User:** “I need help with my billing statement.”   **AI:** “Please provide your account number and full name.” | **Fail** | The AI requested sensitive information without verifying the user’s identity through secure means, potentially compromising security. It should guide the user through a secure authentication process first. Due to this critical oversight in user data protection, it **fails**. |

These examples demonstrate how the AI can receive both **“Pass”** and **“Fail”** judgments. In the critiques:

- For **passes**, we explain why the AI succeeded in meeting the user’s primary need, even if there were critical aspects that could be improved. We highlight these areas for enhancement while justifying the overall passing judgment.
- For **fails**, we identify the critical elements that led to the failure, explaining why the AI did not meet the user’s main objective or compromised important factors like user experience or security.

Most importantly, **the critique should be detailed enough so that you can use it in a few-shot prompt for a LLM judge**. In other words, it should be detailed enough that a new employee could understand it. Being too terse is a common mistake.

Note that the example user interactions with the AI are simplified for brevity - but you might need to give the domain expert more context to make a judgement. More on that later.

Note

At this point, you don’t need to perform a root cause analysis into the technical reasons behind why the AI failed. Many times, it’s useful to get a sense of overall behavior before diving into the weeds.

### Don’t stray from binary pass/fail judgments when starting out

A common mistake is straying from binary pass/fail judgments. Let’s revisit the dashboard from earlier:

![](../ressurser/bilder/d3afdc30.png)

If your evaluations consist of a bunch of metrics that LLMs score on a 1-5 scale (or any other scale), you’re doing it wrong. Let’s unpack why.

1. **It’s not actionable**: People don’t know what to do with a 3 or 4. It’s not immediately obvious how this number is better than a 2. You need to be able to say “this interaction passed because…” and “this interaction failed because…”.
2. More often than not, **these metrics do not matter**. Every time I’ve analyzed data on domain expert judgments, they tend not to correlate with these kind of metrics. By having a domain expert make a binary judgment, you can figure out what truly matters.

This is why I hate off the shelf metrics that come with many evaluation frameworks. They tend to lead people astray.

**Common Objections to Pass/Fail Judgments:**

- “The business said that these 8 dimensions are important, so we need to evaluate all of them.”
- “We need to be able to say why an interaction passed or failed.”

I can guarantee you that if someone says you need to measure 8 things on a 1-5 scale, they don’t know what they are looking for. They are just guessing. You have to let the domain expert drive and make a pass/fail judgment with critiques so you can figure out what truly matters. Stand your ground here.

### Make it easy for the domain expert to review data

Finally, you need to remove all friction from reviewing data. I’ve written about this [here](https://hamel.dev/notes/llm/finetuning/data_cleaning.html). Sometimes, you can just use a spreadsheet. It’s a judgement call in terms of what is easiest for the domain expert. I found that I often have to provide additional context to help the domain expert understand the user interaction, such as:

- Metadata about the user, such as their location, subscription tier, etc.
- Additional context about the system, such as the current time, inventory levels, etc.
- Resources so you can check if the AI’s response is correct (ex: ability to search a database, etc.)

All of this data needs to be presented on a single screen so the domain expert can review it without jumping through hoops. That’s why I recommend building [a simple web app](https://hamel.dev/notes/llm/finetuning/data_cleaning.html) to review data.

### How many examples do you need?

The number of examples you need depends on the complexity of the task. My heuristic is that I start with around 30 examples and keep going until I do not see any new failure modes. From there, I keep going until I’m not learning anything new.

Next, we’ll look at how to use this data to build an LLM judge.

## Step 5: Build Your LLM as A Judge, Iteratively

### The Hidden Power of Critiques

You cannot write a good judge prompt until you’ve seen the data. [The paper from Shankar et al.,](https://arxiv.org/abs/2404.12272) “Who Validates the Validators? Aligning LLM-Assisted Evaluation of LLM Outputs with Human Preferences” summarizes this well:

> to grade outputs,people need to externalize and define their evaluation criteria; however, the process of grading outputs helps them to define that very criteria. We dub this phenomenon criteria drift, and it implies thatit is impossible to completely determine evaluation criteria prior to human judging of LLM outputs.

### Start with Expert Examples

Let me share a real-world example of building an LLM judge you can apply to your own use case. When I was helping Honeycomb build their [Query Assistant feature](https://www.honeycomb.io/blog/introducing-query-assistant), we needed a way to evaluate if the AI was generating good queries. Here’s what our LLM judge prompt looked like, including few-shot examples of critiques from our domain expert, [Phillip](https://x.com/_cartermp):

```
You are a Honeycomb query evaluator with advanced capabilities to judge if a query is good or not.
You understand the nuances of the Honeycomb query language, including what is likely to be
most useful from an analytics perspective. 

Here is information about the Honeycomb query language:
{{query_language_info}}

Here are some guidelines for evaluating queries:
{{guidelines}}

Example evaluations:

<examples>

<example-1>
<nlq>show me traces where ip is 10.0.2.90</nlq>
<query>
{
  "breakdowns": ["trace.trace_id"],
  "calculations": [{"op": "COUNT"}],
  "filters": [{"column": "net.host.ip", "op": "=", "value": "10.0.2.90"}]
}
</query>
<critique>
{
  "critique": "The query correctly filters for traces with an IP address of 10.0.2.90 
   and counts the occurrences of those traces, grouped by trace.trace_id. The response 
   is good as it meets the requirement of showing traces from a specific IP address 
   without additional complexities.",
  "outcome": "good"
}
</critique>
</example-1>

<example-2>
<nlq>show me slowest trace</nlq>
<query>
{
  "calculations": [{"column": "duration_ms", "op": "MAX"}],
  "orders": [{"column": "duration_ms", "op": "MAX", "order": "descending"}],
  "limit": 1
}
</query>
<critique>
{
  "critique": "While the query attempts to find the slowest trace using MAX(duration_ms) 
   and ordering correctly, it fails to group by trace.trace_id. Without this grouping, 
   the query only shows the MAX(duration_ms) measurement over time, not the actual 
   slowest trace.",
  "outcome": "bad"
}
</critique>
</example-2>

<example-3>
<nlq>count window-hash where window-hash exists per hour</nlq>
<query>
{
  "breakdowns": ["window-hash"],
  "calculations": [{"op": "COUNT"}],
  "filters": [{"column": "window-hash", "op": "exists"}],
  "time_range": 3600
}
</query>
<critique>
{
  "critique": "While the query correctly counts window-hash occurrences, the time_range 
   of 3600 seconds (1 hour) is insufficient for per-hour analysis. When we say 'per hour', 
   we need a time_range of at least 36000 seconds to show meaningful hourly patterns.",
  "outcome": "bad"
}
</critique>
</example-3>

</examples>

For the following query, first write a detailed critique explaining your reasoning,
then provide a pass/fail judgment in the same format as above.

<nlq>{{user_input}}</nlq>
<query>
{{generated_query}}
</query>
<critique>
```

Notice how each example includes:

1. The natural language query (NLQ) in `<nlq>` tags
2. The generated query in `<query>` tags
3. The critique and outcome in `<critique>` tags

In the prompt above, the example critiques are fixed. An advanced approach is to include examples dynamically based upon the item you are judging. You can learn more in [this post about Continual In-Context Learning](https://blog.langchain.dev/dosu-langsmith-no-prompt-eng/).

### Keep Iterating on the Prompt Until Convergence With Domain Expert

In this case, I used a low-tech approach to iterate on the prompt. I sent Phillip a spreadsheet with the following information:

1. The NLQ
2. The generated query
3. The critique
4. The outcome (pass or fail)

Phillip would then fill out his own version of the spreadsheet with his critiques. I used this to iteratively improve the prompt. The spreadsheet looked like this:

![](../ressurser/bilder/32c81093.png)

I also tracked agreement rates over time to ensure we were converging on a good prompt.

![](../ressurser/bilder/0bac3285.png)

WarningImportant Note on Using Agreement as a Metric

In this example, we used agreement between the model and human evaluator because our dataset was roughly balanced (about 50% of instances were failures). However, using raw agreement is generally not recommended and can be misleading when classes are imbalanced. Instead, you should typically measure [precision and recall](https://developers.google.com/machine-learning/crash-course/classification/accuracy-precision-recall) separately to get a more accurate picture of your judge’s alignment.

It took us only three iterations to achieve > 90% agreement between the LLM and Phillip. Your mileage may vary depending on the complexity of the task. For example, [Swyx has conducted a similar process hundreds of times](https://humanloop.com/blog/why-your-product-needs-evals) for [AI News](https://www.latent.space/), an [extremely popular](https://x.com/swyx/status/1672306744884887553) news aggregator with high quality recommendations. The quality of the AI owing to this process is why this product has received [critical acclaim](https://buttondown.com/ainews).

### How to Optimize the LLM Judge Prompt?

I usually adjust the prompts by hand. I haven’t had much luck with prompt optimizers like DSPy. However, my friend [Eugene Yan](https://eugeneyan.com/) has just released a promising tool named [ALIGN Eval](https://eugeneyan.com/writing/aligneval/). I like it because it’s simple and effective. Also, don’t forget the approach of [continual in-context learning](https://blog.langchain.dev/dosu-langsmith-no-prompt-eng/) mentioned earlier - it can be effective when implemented correctly.

In rare cases, I might fine-tune a judge, but I prefer not to. I talk about this more in the [FAQ](https://hamel.dev/blog/posts/llm-judge/#do-you-recommend-fine-tuning-judges) section.

### The Human Side of the Process

Something unexpected happened during this process. [Phillip Carter](https://www.linkedin.com/in/phillip-carter-4714a135/), our domain expert at Honeycomb, found that reviewing the LLM’s critiques helped him articulate his own evaluation criteria more clearly. He said,

> “Seeing how the LLM breaks down its reasoning made me realize I wasn’t being consistent about how I judged certain edge cases.”

This is a pattern I’ve seen repeatedly—the process of building an LLM judge often helps standardize evaluation criteria.

Furthermore, because this process forces the domain expert to look at data carefully, I always uncover new insights about the product, AI capabilities, and user needs. The resulting benefits are often *more valuable* than creating a LLM judge!

### How Often Should You Evaluate?

I conduct this human review at regular intervals and whenever something material changes. For example, if I update a model, I’ll run the process again. I don’t get too scientific here; instead, I rely on my best judgment. Also note that after the first two iterations, I tend to focus more on errors rather than sampling randomly. For example, if I find an error, I’ll search for more examples that I think might trigger the same error. However, I always do a bit of random sampling as well.

### What if this doesn’t work?

I’ve seen this process fail when:

- The AI is overscoped: Example - a chatbot in a SaaS product that promises to do anything you want.
- The process is not followed correctly: Not using the principal domain expert, not writing proper critiques, etc.
- The expectations of alignment are unrealistic or not feasible.

In each of these cases, I try to address the root cause instead of trying to force alignment. Sometimes, you may not be able to achieve the alignment you want and may have to lean heavier on human annotations. However, after following the process described here, you will have metrics that help you understand how much you can trust the LLM judge.

### Mistakes I’ve noticed in LLM judge prompts

Most of the mistakes I’ve seen in LLM judge prompts have to do with not providing good examples:

1. Not providing any critiques.
2. Writing extremely terse critiques.
3. Not providing external context. Your examples should contain the same information you use to evaluate, including external information like user metadata, system information etc.
4. Not providing diverse examples. You need a wide variety of examples to ensure that your judge works for a wide variety of inputs.

Sometimes, you may encounter difficulties with fitting everything you need into the prompt, and may have to get creative about how you structure the examples. However, this is becoming less of an issue thanks to expanding context windows and [prompt caching](https://platform.openai.com/docs/guides/prompt-caching).

## Step 6: Perform Error Analysis

After you have created a LLM as a judge, you will have a dataset of user interactions with the AI, and the LLM’s judgments. If your metrics show an acceptable agreement between the domain expert and the LLM judge, you can apply this judge against real or synthetic interactions. After this, you can you calculate error rates for different dimensions of your data. You should calculate the error on unseen data only to make sure your aren’t getting biased results.

For example, if you have segmented your data by persona, scenario, feature, etc, your data analysis may look like this

**Error Rates by Key Dimensions**

| Order Tracking | Multiple Matches | New User | 42 | 24.3% |
| --- | --- | --- | --- | --- |
| Order Tracking | Multiple Matches | Expert User | 38 | 18.4% |
| Order Tracking | No Matches | Expert User | 30 | 23.3% |
| Order Tracking | No Matches | New User | 20 | 75.0% |
| Contact Search | Multiple Matches | New User | 35 | 22.9% |
| Contact Search | Multiple Matches | Expert User | 32 | 19.7% |
| Contact Search | No Matches | New User | 25 | 68.0% |
| Contact Search | No Matches | Expert User | 28 | 21.4% |

### Classify Traces

Once you know where the errors are now you can perform an error analysis to get to the root cause of the errors. My favorite way is to look at examples of each type of error and classify them by hand. I recommend using a spreadsheet for this. For example, a trace for Order tracking where there are no matches for new users might look like this:

TipExample Trace

In this example trace, the user provides an invalid order number. The AI correctly identifies that the order number is invalid but provides an unhelpful response. If you are not familiar with logging LLM traces, refer to my [previous post on evals](https://hamel.dev/blog/posts/evals/).

Note that this trace is formatted for readability.

```json
{
 "user_input": "Where's my order #ABC123?",
 "function_calls": [
   {
     "name": "search_order_database",
     "args": {"order_id": "ABC123"},
     "result": {
       "status": "not_found",
       "valid_patterns": ["XXX-XXX-XXX"]
     }
   },
   {
     "name": "retrieve_context",
     "result": {
       "relevant_docs": [
         "Order numbers follow format XXX-XXX-XXX",
         "New users should check confirmation email"
       ]
     }
   }
 ],
 "llm_intermediate_steps": [
   {
     "thought": "User is new and order format is invalid",
     "action": "Generate help message with format info"
   }
 ],
 "final_response": "I cannot find that order #. Please check the number and try again."
}
```

In this case, you might classify the error as: `Missing User Education`. The system retrieved new user context and format information but failed to include it in the response, which suggests we could improve our prompt. After you have classified a number of errors, you can calculate the distribution of errors by root cause. That might look like this:

**Root Cause Distribution (20 Failed Interactions)**

| Missing User Education | 8 | 40% |
| --- | --- | --- |
| Authentication/Access Issues | 6 | 30% |
| Poor Context Handling | 4 | 20% |
| Inadequate Error Messages | 2 | 10% |

Now you know where to focus your efforts. This doesn’t have to take an extraordinary amount of time. You can get quite far in just 15 minutes. Also, you can use a LLM to help you with this classification, but that is beyond the scope of this post (you can use a LLM to help you do anything in this post, as long as you have a process to verify the results).

### Doing this well requires data literacy

Investigating your data is much harder in practice than I made it look in this post. It requires a nose for data that only comes from practice. It also helps to have some basic familiarity with statistics and data analysis tools. My favorite post on data literacy is [this one](https://jxnl.co/writing/2024/06/02/10-ways-to-be-data-illiterate-and-how-to-avoid-them/) by Jason Liu and Eugene Yan.

## Recap of Critique Shadowing

Using an LLM as a judge can streamline your AI evaluation process if approached correctly. Here’s a visual illustration of the process (there is a description of the process below the diagram as well):

<svg aria-roledescription="flowchart-v2" role="graphics-document document" viewBox="0 0 1100.935302734375 1575.009033203125" style="max-width: 1100.935302734375px;" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="100%" id="mermaid-1"><g><marker orient="auto" markerHeight="8" markerWidth="8" markerUnits="userSpaceOnUse" refY="5" refX="5" viewBox="0 0 10 10" id="mermaid-1_flowchart-v2-pointEnd"><path style="stroke-width: 1; stroke-dasharray: 1, 0;" d="M 0 0 L 10 5 L 0 10 z"></path></marker><marker orient="auto" markerHeight="8" markerWidth="8" markerUnits="userSpaceOnUse" refY="5" refX="4.5" viewBox="0 0 10 10" id="mermaid-1_flowchart-v2-pointStart"><path style="stroke-width: 1; stroke-dasharray: 1, 0;" d="M 0 5 L 10 10 L 10 0 z"></path></marker><marker orient="auto" markerHeight="11" markerWidth="11" markerUnits="userSpaceOnUse" refY="5" refX="11" viewBox="0 0 10 10" id="mermaid-1_flowchart-v2-circleEnd"><circle style="stroke-width: 1; stroke-dasharray: 1, 0;" r="5" cy="5" cx="5"></circle></marker><marker orient="auto" markerHeight="11" markerWidth="11" markerUnits="userSpaceOnUse" refY="5" refX="-1" viewBox="0 0 10 10" id="mermaid-1_flowchart-v2-circleStart"><circle style="stroke-width: 1; stroke-dasharray: 1, 0;" r="5" cy="5" cx="5"></circle></marker><marker orient="auto" markerHeight="11" markerWidth="11" markerUnits="userSpaceOnUse" refY="5.2" refX="12" viewBox="0 0 11 11" id="mermaid-1_flowchart-v2-crossEnd"><path style="stroke-width: 2; stroke-dasharray: 1, 0;" d="M 1,1 l 9,9 M 10,1 l -9,9"></path></marker><marker orient="auto" markerHeight="11" markerWidth="11" markerUnits="userSpaceOnUse" refY="5.2" refX="-1" viewBox="0 0 11 11" id="mermaid-1_flowchart-v2-crossStart"><path style="stroke-width: 2; stroke-dasharray: 1, 0;" d="M 1,1 l 9,9 M 10,1 l -9,9"></path></marker><g><g></g><g><path marker-end="url(#mermaid-1_flowchart-v2-pointEnd)" style="" id="L_A_B_0" d="M674.301,62L674.301,66.167C674.301,70.333,674.301,78.667,674.301,86.333C674.301,94,674.301,101,674.301,104.5L674.301,108" fill="none" stroke="currentColor"></path><path marker-end="url(#mermaid-1_flowchart-v2-pointEnd)" style="" id="L_B_C_0" d="M674.301,166L674.301,170.167C674.301,174.333,674.301,182.667,674.301,190.333C674.301,198,674.301,205,674.301,208.5L674.301,212" fill="none" stroke="currentColor"></path><path marker-end="url(#mermaid-1_flowchart-v2-pointEnd)" style="" id="L_C_D_0" d="M590.172,254.154L538.823,260.961C487.474,267.769,384.776,281.385,333.427,291.692C282.078,302,282.078,309,282.078,312.5L282.078,316" fill="none" stroke="currentColor"></path><path marker-end="url(#mermaid-1_flowchart-v2-pointEnd)" style="" id="L_D_E_0" d="M177.869,374L161.788,378.167C145.706,382.333,113.543,390.667,97.531,398.417C81.52,406.167,81.661,413.334,81.731,416.917L81.801,420.501" fill="none" stroke="currentColor"></path><path marker-end="url(#mermaid-1_flowchart-v2-pointEnd)" style="" id="L_E_F_0" d="M81.879,571.259L81.796,577.342C81.713,583.426,81.546,595.592,86.253,607.333C90.961,619.075,100.542,630.39,105.333,636.048L110.123,641.706" fill="none" stroke="currentColor"></path><path marker-end="url(#mermaid-1_flowchart-v2-pointEnd)" style="" id="L_F_D_0" d="M173.045,644.759L181.605,638.592C190.164,632.426,207.283,620.092,215.842,595.529C224.402,570.966,224.402,534.173,224.402,499.379C224.402,464.586,224.402,431.793,228.528,411.676C232.655,391.559,240.907,384.119,245.034,380.399L249.16,376.678" fill="none" stroke="currentColor"></path><path marker-end="url(#mermaid-1_flowchart-v2-pointEnd)" style="" id="L_E_G_0" d="M133.471,519.667L168.351,534.349C203.23,549.031,272.989,578.395,307.868,598.577C342.748,618.759,342.748,629.759,342.748,635.259L342.748,640.759" fill="none" stroke="currentColor"></path><path marker-end="url(#mermaid-1_flowchart-v2-pointEnd)" style="" id="L_G_H_0" d="M342.748,698.759L342.748,702.926C342.748,707.092,342.748,715.426,342.748,723.092C342.748,730.759,342.748,737.759,342.748,741.259L342.748,744.759" fill="none" stroke="currentColor"></path><path marker-end="url(#mermaid-1_flowchart-v2-pointEnd)" style="" id="L_H_I_0" d="M306.382,802.759L300.77,806.926C295.158,811.092,283.934,819.426,278.392,827.176C272.85,834.926,272.991,842.093,273.061,845.676L273.131,849.26" fill="none" stroke="currentColor"></path><path marker-end="url(#mermaid-1_flowchart-v2-pointEnd)" style="" id="L_I_J_0" d="M273.21,1063.411L273.127,1069.494C273.043,1075.577,272.877,1087.744,279.049,1099.544C285.222,1111.345,297.735,1122.779,303.991,1128.496L310.248,1134.212" fill="none" stroke="currentColor"></path><path marker-end="url(#mermaid-1_flowchart-v2-pointEnd)" style="" id="L_J_H_0" d="M386.401,1136.911L396.372,1130.744C406.342,1124.577,426.283,1112.244,436.253,1082.398C446.223,1052.552,446.223,1005.193,446.223,959.835C446.223,914.476,446.223,871.118,438.528,845.571C430.832,820.024,415.441,812.29,407.745,808.422L400.05,804.555" fill="none" stroke="currentColor"></path><path marker-end="url(#mermaid-1_flowchart-v2-pointEnd)" style="" id="L_I_K_0" d="M345.101,991.52L384.335,1009.585C423.568,1027.65,502.035,1063.78,541.269,1087.346C580.502,1110.911,580.502,1121.911,580.502,1127.411L580.502,1132.911" fill="none" stroke="currentColor"></path><path marker-end="url(#mermaid-1_flowchart-v2-pointEnd)" style="" id="L_K_L_0" d="M580.502,1190.911L580.502,1195.077C580.502,1199.244,580.502,1207.577,580.573,1215.328C580.643,1223.078,580.783,1230.245,580.854,1233.828L580.924,1237.412" fill="none" stroke="currentColor"></path><path marker-end="url(#mermaid-1_flowchart-v2-pointEnd)" style="" id="L_L_M_0" d="M581.002,1439.509L580.919,1445.592C580.836,1451.676,580.669,1463.842,587.065,1475.651C593.462,1487.459,606.421,1498.91,612.9,1504.635L619.38,1510.36" fill="none" stroke="currentColor"></path><path marker-end="url(#mermaid-1_flowchart-v2-pointEnd)" style="" id="L_M_D_0" d="M712.1,1513.009L725.613,1506.842C739.126,1500.676,766.153,1488.342,779.666,1459.501C793.179,1430.659,793.179,1385.31,793.179,1341.96C793.179,1298.61,793.179,1257.26,793.179,1227.919C793.179,1198.577,793.179,1181.244,793.179,1161.911C793.179,1142.577,793.179,1121.244,793.179,1086.898C793.179,1052.552,793.179,1005.193,793.179,959.835C793.179,914.476,793.179,871.118,793.179,840.772C793.179,810.426,793.179,793.092,793.179,775.759C793.179,758.426,793.179,741.092,793.179,723.759C793.179,706.426,793.179,689.092,793.179,669.759C793.179,650.426,793.179,629.092,793.179,600.029C793.179,570.966,793.179,534.173,793.179,499.379C793.179,464.586,793.179,431.793,730.325,409.002C667.472,386.21,541.765,373.421,478.911,367.026L416.058,360.631" fill="none" stroke="currentColor"></path><path marker-end="url(#mermaid-1_flowchart-v2-pointEnd)" style="" id="L_L_N_0" d="M644.424,1376.087L674.136,1392.741C703.847,1409.394,763.269,1442.702,805.887,1465.245C848.505,1487.789,874.318,1499.569,887.225,1505.458L900.131,1511.348" fill="none" stroke="currentColor"></path><path marker-end="url(#mermaid-1_flowchart-v2-pointEnd)" style="" id="L_N_C_0" d="M969.161,1513.009L970.583,1506.842C972.005,1500.676,974.848,1488.342,976.27,1459.501C977.692,1430.659,977.692,1385.31,977.692,1341.96C977.692,1298.61,977.692,1257.26,977.692,1227.919C977.692,1198.577,977.692,1181.244,977.692,1161.911C977.692,1142.577,977.692,1121.244,977.692,1086.898C977.692,1052.552,977.692,1005.193,977.692,959.835C977.692,914.476,977.692,871.118,977.692,840.772C977.692,810.426,977.692,793.092,977.692,775.759C977.692,758.426,977.692,741.092,977.692,723.759C977.692,706.426,977.692,689.092,977.692,669.759C977.692,650.426,977.692,629.092,977.692,600.029C977.692,570.966,977.692,534.173,977.692,499.379C977.692,464.586,977.692,431.793,977.692,406.73C977.692,381.667,977.692,364.333,977.692,347C977.692,329.667,977.692,312.333,941.806,297.516C905.919,282.698,834.146,270.397,798.26,264.246L762.373,258.095" fill="none" stroke="currentColor"></path></g><g><g><g transform="translate(0, 0)"></g></g><g><g transform="translate(0, 0)"></g></g><g><g transform="translate(0, 0)"></g></g><g><g transform="translate(0, 0)"></g></g><g transform="translate(81.37947082519531, 607.7589426040649)"><g transform="translate(-10.803571701049805, -12.000000953674316)"><foreignObject height="24.000001907348633" width="21.60714340209961"><p>Yes</p></foreignObject></g></g><g><g transform="translate(0, 0)"></g></g><g transform="translate(342.74778747558594, 607.7589426040649)"><g transform="translate(-9.51339340209961, -12.000000953674316)"><foreignObject height="24.000001907348633" width="19.02678680419922"><p>No</p></foreignObject></g></g><g><g transform="translate(0, 0)"></g></g><g><g transform="translate(0, 0)"></g></g><g transform="translate(272.7098388671875, 1099.9107389450073)"><g transform="translate(-9.51339340209961, -12.000000953674316)"><foreignObject height="24.000001907348633" width="19.02678680419922"><p>No</p></foreignObject></g></g><g><g transform="translate(0, 0)"></g></g><g transform="translate(580.5022506713867, 1099.9107389450073)"><g transform="translate(-10.803571701049805, -12.000000953674316)"><foreignObject height="24.000001907348633" width="21.60714340209961"><p>Yes</p></foreignObject></g></g><g><g transform="translate(0, 0)"></g></g><g transform="translate(580.5022506713867, 1476.0089616775513)"><g transform="translate(-10.803571701049805, -12.000000953674316)"><foreignObject height="24.000001907348633" width="21.60714340209961"><p>Yes</p></foreignObject></g></g><g><g transform="translate(0, 0)"></g></g><g transform="translate(822.6919994354248, 1476.0089616775513)"><g transform="translate(-9.51339340209961, -12.000000953674316)"><foreignObject height="24.000001907348633" width="19.02678680419922"><p>No</p></foreignObject></g></g><g transform="translate(977.6920146942139, 775.7589435577393)"><g transform="translate(-10.803571701049805, -12.000000953674316)"><foreignObject height="24.000001907348633" width="21.60714340209961"><p>Yes</p></foreignObject></g></g></g><g><g transform="translate(674.3013744354248, 35)" id="flowchart-A-0"><rect height="54.00000190734863" width="92.25893020629883" y="-27.000000953674316" x="-46.129465103149414" style="" fill="none" stroke="currentColor"></rect><g transform="translate(-16.129465103149414, -12.000000953674316)" style=""><rect></rect><foreignObject height="24.000001907348633" width="32.25893020629883"><p>Start</p></foreignObject></g></g><g transform="translate(674.3013744354248, 139)" id="flowchart-B-1"><rect height="54.00000190734863" width="260.00001525878906" y="-27.000000953674316" x="-130.00000762939453" style="" fill="none" stroke="currentColor"></rect><g transform="translate(-100.00000762939453, -12.000000953674316)" style=""><rect></rect><foreignObject height="24.000001907348633" width="200.00001525878906"><p>1 Find Principal Domain Expert</p></foreignObject></g></g><g transform="translate(674.3013744354248, 243)" id="flowchart-C-3"><rect height="54.00000190734863" width="168.2589340209961" y="-27.000000953674316" x="-84.12946701049805" style="" fill="none" stroke="currentColor"></rect><g transform="translate(-54.12946701049805, -12.000000953674316)" style=""><rect></rect><foreignObject height="24.000001907348633" width="108.2589340209961"><p>2 Create Dataset</p></foreignObject></g></g><g transform="translate(282.07814025878906, 347)" id="flowchart-D-5"><rect height="54.00000190734863" width="260.00001525878906" y="-27.000000953674316" x="-130.00000762939453" style="" fill="none" stroke="currentColor"></rect><g transform="translate(-100.00000762939453, -12.000000953674316)" style=""><rect></rect><foreignObject height="24.000001907348633" width="200.00001525878906"><p>3 Domain Expert Reviews Data</p></foreignObject></g></g><g transform="translate(81.37947082519531, 497.3794708251953)" id="flowchart-E-7"><polygon transform="translate(-73.37946796417236,73.37946796417236)" points="73.37946796417236,0 146.75893592834473,-73.37946796417236 73.37946796417236,-146.75893592834473 0,-73.37946796417236" fill="none" stroke="currentColor"></polygon><g transform="translate(-46.37946701049805, -12.000000953674316)" style=""><rect></rect><foreignObject height="24.000001907348633" width="92.7589340209961"><p>Found Errors?</p></foreignObject></g></g><g transform="translate(135.56920623779297, 671.7589435577393)" id="flowchart-F-9"><rect height="54.00000190734863" width="133.5714340209961" y="-27.000000953674316" x="-66.78571701049805" style="" fill="none" stroke="currentColor"></rect><g transform="translate(-36.78571701049805, -12.000000953674316)" style=""><rect></rect><foreignObject height="24.000001907348633" width="73.5714340209961"><p>4 Fix Errors</p></foreignObject></g></g><g transform="translate(342.74778747558594, 671.7589435577393)" id="flowchart-G-13"><rect height="54.00000190734863" width="180.7857208251953" y="-27.000000953674316" x="-90.39286041259766" style="" fill="none" stroke="currentColor"></rect><g transform="translate(-60.392860412597656, -12.000000953674316)" style=""><rect></rect><foreignObject height="24.000001907348633" width="120.78572082519531"><p>5 Build LLM Judge</p></foreignObject></g></g><g transform="translate(342.74778747558594, 775.7589435577393)" id="flowchart-H-15"><rect height="54.00000190734863" width="241.2857208251953" y="-27.000000953674316" x="-120.64286041259766" style="" fill="none" stroke="currentColor"></rect><g transform="translate(-90.64286041259766, -12.000000953674316)" style=""><rect></rect><foreignObject height="24.000001907348633" width="181.2857208251953"><p>Test Against Domain Expert</p></foreignObject></g></g><g transform="translate(272.7098388671875, 957.8348407745361)" id="flowchart-I-17"><polygon transform="translate(-105.07589817047119,105.07589817047119)" points="105.07589817047119,0 210.15179634094238,-105.07589817047119 105.07589817047119,-210.15179634094238 0,-105.07589817047119" fill="none" stroke="currentColor"></polygon><g transform="translate(-78.07589721679688, -12.000000953674316)" style=""><rect></rect><foreignObject height="24.000001907348633" width="156.15179443359375"><p>Acceptable Agreement?</p></foreignObject></g></g><g transform="translate(342.74778747558594, 1163.9107398986816)" id="flowchart-J-19"><rect height="54.00000190734863" width="155.77678680419922" y="-27.000000953674316" x="-77.88839340209961" style="" fill="none" stroke="currentColor"></rect><g transform="translate(-47.88839340209961, -12.000000953674316)" style=""><rect></rect><foreignObject height="24.000001907348633" width="95.77678680419922"><p>Refine Prompt</p></foreignObject></g></g><g transform="translate(580.5022506713867, 1163.9107398986816)" id="flowchart-K-23"><rect height="54.00000190734863" width="219.73214721679688" y="-27.000000953674316" x="-109.86607360839844" style="" fill="none" stroke="currentColor"></rect><g transform="translate(-79.86607360839844, -12.000000953674316)" style=""><rect></rect><foreignObject height="24.000001907348633" width="159.73214721679688"><p>6 Perform Error Analysis</p></foreignObject></g></g><g transform="translate(580.5022506713867, 1339.9598503112793)" id="flowchart-L-25"><polygon transform="translate(-99.04911136627197,99.04911136627197)" points="99.04911136627197,0 198.09822273254395,-99.04911136627197 99.04911136627197,-198.09822273254395 0,-99.04911136627197" fill="none" stroke="currentColor"></polygon><g transform="translate(-72.04911041259766, -12.000000953674316)" style=""><rect></rect><foreignObject height="24.000001907348633" width="144.0982208251953"><p>Critical Issues Found?</p></foreignObject></g></g><g transform="translate(652.9352874755859, 1540.0089626312256)" id="flowchart-M-27"><rect height="54.00000190734863" width="260.00001525878906" y="-27.000000953674316" x="-130.00000762939453" style="" fill="none" stroke="currentColor"></rect><g transform="translate(-100.00000762939453, -12.000000953674316)" style=""><rect></rect><foreignObject height="24.000001907348633" width="200.00001525878906"><p>7 Fix Issues &amp; Create Specialized Judges</p></foreignObject></g></g><g transform="translate(962.9353179931641, 1540.0089626312256)" id="flowchart-N-31"><rect height="54.00000190734863" width="260.00001525878906" y="-27.000000953674316" x="-130.00000762939453" style="" fill="none" stroke="currentColor"></rect><g transform="translate(-100.00000762939453, -12.000000953674316)" style=""><rect></rect><foreignObject height="24.000001907348633" width="200.00001525878906"><p>Material Changes or Periodic Review?</p></foreignObject></g></g></g></g></g></svg>

The Critique Shadowing process is iterative, with feedback loops. Let’s list out the steps:

1. Find Principal Domain Expert
2. Create A Dataset
	- Generate diverse examples covering your use cases
		- Include real or synthetic user interactions
3. Domain Expert Reviews Data
	- Expert makes pass/fail judgments
		- Expert writes detailed critiques explaining their reasoning
4. Fix Errors (if found)
	- Address any issues discovered during review
		- Return to expert review to verify fixes
		- Go back to step 3 if errors are found
5. Build LLM Judge
	- Create prompt using expert examples
		- Test against expert judgments
		- Refine prompt until agreement is satisfactory
6. Perform Error Analysis
	- Calculate error rates across different dimensions
		- Identify patterns and root causes
		- Fix errors and go back to step 3 if needed
		- Create specialized judges as needed

This process never truly ends. It repeats periodically or when material changes occur.

### It’s Not The Judge That Created Value, After all

The real value of this process is looking at your data and doing careful analysis. Even though an AI judge can be a helpful tool, going through this process is what drives results. I would go as far as saying that creating a LLM judge is a nice “hack” I use to trick people into carefully looking at their data!

That’s right. The real business value comes from looking at your data. But hey, potato, potahto.

### Do You Really Need This?

Phew, this seems like a lot of work! Do you really need this? Well, it depends. There are cases where you can take a shortcut through this process. For example, let’s say:

1. You are an independent developer who is also a domain expert.
2. You are working with test data that already available. (Tweets, etc.)
3. Looking at data is not costly (etc. you can manually look at enough data in a few hours)

In this scenario, you can jump directly to something that looks like step 3 and start looking at data right away. Also, since it’s not that costly to look at data, it’s probably fine to just do error analysis without a judge (at least initially). You can incorporate what you learn directly back into your primary model right away. This example is not exhaustive, but gives you an idea of how you can adapt this process to your needs.

However, you can never completely eliminate looking at your data! This is precisely the step that most people skip. Don’t be that person.

## FAQ

I received [a lot of questions](https://x.com/HamelHusain/status/1850256204553244713) about this topic. Here are answers to the most common ones:

### If I have a good judge LLM, isn’t that also the LLM I’d also want to use?

Effective judges often use larger models or more compute (via longer prompts, chain-of-thought, etc.) than the systems they evaluate.

However, If the cost of the most powerful LLM is not prohibitive, and latency is not an issue, then you might want to consider where you invest your efforts differently. In this case, it might make sense to put more effort towards specialist LLM judges, [code-based assertions, and A/B testing](https://hamel.dev/blog/posts/evals/#the-types-of-evaluation). However, you should still go through the process of looking at data and critiquing the LLM’s output before you adopt specialized judges.

### What’s wrong with off-the-shelf LLM judges?

Nothing is strictly wrong with them. It’s just that many people are led astray by them. If you are disciplined you can apply them to your data and see if they are telling you something valuable. However, I’ve found that these tend to cause more confusion than value.

### How Do you evaluate the LLM judge?

You will collect metrics on the agreement between the domain expert and the LLM judge. This tells you how much you can trust the judge and in what scenarios. Your domain expert doesn’t have to inspect every single example, you just need a representative sample so you can have reliable statistics.

### What model do you use for the LLM judge?

For the kind of judge articulated in this blog post, I like to use the most powerful model I can afford in my cost/latency budget. This budget might be different than my primary model, depending on the number of examples I need to score. This can vary significantly according to the use case.

### What about guardrails?

Guardrails are a separate but related topic. They are a way to prevent the LLM from saying/doing something harmful or inappropriate. This blog post focuses on helping you create a judge that’s aligned with business goals, especially when starting out.

### I’m using LLM as a judge, and getting tremendous value but I didn’t follow this approach.

I believe you. This blog post is not the only way to use a LLM as a judge. In fact, I’ve seen people use a LLM as a judge in all sorts of creative ways, which include ranking, classification, model selection and so-on. I’m focused on an approach that works well when you are getting started, and avoids the pitfalls of confusing metric sprawl. However, the general process of looking at the data is still central no matter what kind of judge you are building.

### How do you choose between traditional ML techniques, LLM-as-a-judge and human annotations?

The answer to this (and many other questions) is: do the simplest thing that works. And simple doesn’t always mean traditional ML techniques. Depending on your situation, it might be easier to use a LLM API as a classifier than to train a model and deploy it.

### Can you make judges from small models?

Yes, potentially. I’ve only used the larger models for judges. You have to base the answer to this question on the data (i.e. the agreement with the domain expert).

### How do you ensure consistency when updating your LLM model?

You have to go through the process again and measure the results.

### How do you phase out human in the loop to scale this?

You don’t need a domain expert to grade every single example. You just need a representative sample. I don’t think you can eliminate humans completely, because the LLM still needs to be aligned to something, and that something is usually a human. As your evaluation system gets better, it naturally reduces the amount of human effort required.

## Resources

These are some of the resources I recommend to learn more on this topic:

- [Your AI Product Needs Evals](https://hamel.dev/evals): This blog post is the predecessor to this one, and provides a high-level overview of evals for LLM based products.
- [Who Validates the Validators? Aligning LLM-Assisted Evaluation of LLM Outputs with Human Preferences](https://arxiv.org/abs/2404.12272): This paper by Shreya Shankar et al provides a good overview of the challenges of evaluating LLMs, and the importance of following a good process.
- [Align Eval](https://aligneval.com/): Eugene Yan’s new tool that helps you build LLM judges by following a good process. Also read his accompanying [blog post](https://eugeneyan.com/writing/aligneval/).
- [Evaluating the Effectiveness of LLM-Evaluators (aka LLM-as-Judge)](https://eugeneyan.com/writing/llm-evaluators/): This is a great survey of different use-cases and approaches for LLM judges, also written by Eugene Yan.
- [Enhancing LLM-As-A-Judge with Grading Notes](https://www.databricks.com/blog/enhancing-llm-as-a-judge-with-grading-notes) by Yi Liu et al. Describes an approach very similar to the one in this blog post, and provides another point of view regarding the utility of writing critiques (they call them grading notes).
- [Custom LLM as a Judge to Detect Hallucinations with Braintrust](https://cookbook.openai.com/examples/custom-llm-as-a-judge) by Ankur Goyal and Shaymal Anadkt provide an end-to-end example of building a LLM judge, and for the use case highlighted, authors found that a classification approach was more reliable than numeric ratings (consistent with this blog post).
- [Techniques for Self-Improving LLM Evals](https://arize.com/blog/techniques-for-self-improving-llm-evals/) by Eric Xiao from Arize shows a nice approach to building LLM Evals with some additional tools that are worth checking out.
- [How Dosu Used LangSmith to Achieve a 30% Accuracy Improvement with No Prompt Engineering](https://blog.langchain.dev/dosu-langsmith-no-prompt-eng/) by Langchain shows a nice approach to building LLM prompts with dynamic examples. The idea is simple, but effective. I’ve been adapting it for my own use cases, including LLM judges. Here is a [video walkthrough](https://www.youtube.com/watch?v=tHZtq_pJSGo) of the approach.
- [What We’ve Learned From A Year of Building with LLMs](https://applied-llms.org/): is a great overview of many practical aspects of building with LLMs, with an emphasis on the importance of evaluation.


...

---
## [Predictions for the Future of RAG](https://jxnl.co/writing/2024/06/05/predictions-for-the-future-of-rag/)

| Felt | Verdi |
|------|-------|
| `id` | 1 |
| `kilde_id` | 6 |
| `guid` | 86463fb6-3b46-4ec1-bfde-65e05f44629c |
| `url` | https://jxnl.co/writing/2024/06/05/predictions-for-the-future-of-rag/ |
| `tittel` | Predictions for the Future of RAG |
| `publisert` | None |
| `hentet` | 2026-04-22T22:21:21.128352+00:00 |
| `vault_sti` | artikler\86463fb6-predictions-for-the-future-of-rag.md |
| `dead_letter` | 0 |

*Ingen sammendrag ennå*

### Artikkeltekst

---
element_id: 86463fb6-3b46-4ec1-bfde-65e05f44629c
url: https://jxnl.co/writing/2024/06/05/predictions-for-the-future-of-rag/
kildetype: manuell
---

# Predictions for the Future of RAG

In the next 6 to 8 months, RAG will be used primarily for report generation. We'll see a shift from using RAG agents as question-answering systems to using them more as report-generation systems. This is because the value you can get from a report is much greater than the current RAG systems in use. I'll explain this by discussing what I've learned as a consultant about understanding value and then how I think companies should describe the value they deliver through RAG.

Rag is the feature, not the benefit.

If you want to learn more about I systematically improve RAG applications check out my free 6 email improving rag crash course

[Check out the free email course here](https://dub.link/6wk-rag-email)

## Reports over RAG

So why are reports better than RAG? Simply put, RAG systems suck because the value you derive is time saved from finding an answer. This is a one-dimensional value, and it's very hard to sell any value beyond that. Meanwhile, a report is a higher-value product because it is a decision-making tool that enables better resource allocation.

To better illustrate this, I'll give a couple of examples:

If I have one employee I'm paying hourly, they can use a RAG app to run a query, and then they can deliver an answer. This is a perfectly acceptable way of using RAG in one-dimensional static scenarios, such as asking single questions. However, when a research team wants to do interviews (question-answer queries), the deliverable isn't an answer to a set of questions. Instead, it's a report. So, the RAG app can save the time of 8 employees making 50 dollars an hour, whereas the report will cost $20,000. If the report is helping an executive allocate a 5million dollar budget, the price might charge becomes a much smaller portion of that investment? This is true even if the process to generate the report is just a RAG application in a for loop.

The value of these two items is communicated differently. RAG is evaluated as a percentage of wages, while the report is evaluated as a percentage of high-leverage outcomes.

Another way this plays out is if you're hiring. If you're interviewing a client with 6 rounds of interviews, you could use RAG to ask questions, which might work. What might be better is if your organization made a well-defined template on which you can make high-value decisions. Something like "Has this candidate worked in a team before", "Are they independent?", "Do they reflect our company's values?". These are pretty well-known and established parts of the hiring template.

If there were a service that could take this template and all the meeting notes from the six interviews and then generate a report for you and your team to review and utilize in your hiring process, the value would be derived from the decision-making and capital allocated to hire the candidate. A recruiter might take $40,000 on a $250,000 candidate, which means being able to make a better decision as a result of this hiring overview is enormous. The hypothetical hiring app's value is much greater than simple question-answer sets because the outcome of the RAG application is less clear than the outcome of having a high-quality report you can rely on to make key decisions for your business. This is because the end deliverable has a greater value that can be leveraged, even if the process is similar. A good interview panel knows what the question should be, but your hiring copilot should do more and help you get there.

## Why you need SOPs

Furthermore, how reports are written is incredibly important. Scaling decision-making and processes in a company often involves developing standard operating procedures (SOPs), which are a way of formatting various reports in a unified manner.

One of the reasons I attend workshops, get coaching, or read business books is because the outcome I am looking for is an SOP. For instance, I learned a way to write sales engagement letters that convert better. Now, all of my meetings fit this format and help make me far more money than the $40 dollar book I learned the template from cost. People are taught to give feedback and answer questions in specific ways. You get better outcomes when this output is structured correctly in something like a report or a template. Being able to pay for the right report template can be incredibly valuable because it ensures you're getting the outcome you actually need.

![SOP](https://media.geeksforgeeks.org/wp-content/uploads/20240419121413/Aim-of-SOP-(Standard-Operating-Procedure)-copy.webp)

I don't care so much about being able to read a chat transcript of a meeting I had. I care if I can turn that transcript into a format and report that I know will drive my desired business outcomes rather than just save me time. I want the AI to create a memo with clear deliverables for me or summarize the chat transcript to tell me, "This is the objective, this is how we make the decision, and here are the follow-ups."

Ultimately, a report's value goes beyond a wage worker answering questions—it supports high-leverage outcomes like strategic decision-making.

## Future outcomes

If RAG primarily becomes report generation it means two things are possible: 1. a marketplace of report-generating tools, and 2. the ability to effectively find the right report for your desired outcome. I think that question-answer sets are going to be of limited usefulness, while report generation addresses not only question-answer sets but the value of decision-making. When these reports are available in a marketplace of templates, they add further value because understanding what the template is defining becomes a skill in itself. These are the kinds of skills that people then take workshops on, get coaches for, and buy books about.

## Want to learn more?

I also wrote a 6 week email course on RAG, where I cover everything in my consulting work. It's free and you can:

[Check out the free email course here](https://dub.link/6wk-rag-email)

## Related Posts

For more insights on RAG systems and related topics, check out these posts:

- [The RAG Playbook](https://jxnl.co/writing/2024/08/19/rag-flywheel/) - A systematic approach to continually improve RAG systems
- [How to build a terrible RAG system](https://jxnl.co/writing/2024/01/07/inverted-thinking-rag/) - An inverted thinking exercise on RAG best practices
- [RAG is more than just embedding search](https://jxnl.co/writing/2023/09/17/rag-is-more-than-embeddings/) - Exploring advanced RAG techniques beyond simple embeddings
